# Bước 8 – Đặc trưng nào lệch giữa 4 mạng, bỏ chúng đi có giúp chuyển miền không?

Lý thuyết từng bước, kèm ví dụ số, nằm trong tài liệu "TADP-IDS – Lý thuyết bước 8" Claude đã gửi.

| Ô | Việc | Thời gian |
|---|---|---|
| 1 | Kiểm tra môi trường và mã nguồn | 1 phút |
| 2 | Bước 1–2: bảng KS (độ lệch) và bảng AUROC đơn biến (khả năng phân biệt) của 77 đặc trưng | 3–5 phút |
| 3 | Bước 3–4: huấn luyện LightGBM trên 5 tập đặc trưng × 12 cặp × 2 cách tiền xử lý | 40–70 phút |
| 4 | Tổng hợp | 1 phút |

**Không cần GPU (card đồ họa)** – chỉ dùng LightGBM. Kết quả lưu vào `MyDrive/TADP_Q1/results/b8_do_lech_dac_trung/`. Nếu Colab ngắt ở Ô 3, chạy lại Ô 3: phần đã xong được bỏ qua.

## Ô 1 – Kiểm tra môi trường và mã nguồn

In [1]:
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
CODE_DIR = "/content/drive/MyDrive/TADP_Q1/code"
sys.path.insert(0, CODE_DIR)
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
try:
    import lightgbm, pyarrow
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "pyarrow"], check=True)
EXPECTED = {"__init__.py": 125, "cic_data.py": 8579, "config.py": 3263, "data.py": 9956, "experts.py": 5614,
            "feature_shift.py": 10692, "metrics.py": 4002, "phase1.py": 9106, "preprocess.py": 3601,
            "shift.py": 3863, "summary.py": 6028}
ok = True
for f, size in EXPECTED.items():
    p = os.path.join(CODE_DIR, "tadp", f)
    real = os.path.getsize(p) if os.path.exists(p) else None
    if real != size:
        ok = False
        print(f"   SAI: {f} – mong đợi {size} byte, thực tế {real}")
print("Mã nguồn:", "ĐẦY ĐỦ, đúng phiên bản bước 8" if ok else "CÓ LỖI – báo lại cho Claude")

Mounted at /content/drive
Mã nguồn: ĐẦY ĐỦ, đúng phiên bản bước 8


## Ô 2 – Bước 1 và 2: bảng KS và bảng AUROC đơn biến

- **KS** (Kolmogorov–Smirnov): mỗi đặc trưng lệch bao nhiêu giữa 2 mạng, từ 0 (giống hệt) đến 1 (tách rời hẳn). Tính cho 6 cặp mạng, trên toàn bộ lưu lượng và trên riêng lưu lượng bình thường.
- **AUROC đơn biến**: nếu chỉ nhìn MỘT đặc trưng, tách tấn công khỏi bình thường tốt đến đâu, trong từng mạng. Trên 0,5: tấn công có giá trị lớn hơn; dưới 0,5: nhỏ hơn.
- **Đảo chiều**: đặc trưng có AUROC < 0,45 ở mạng này nhưng > 0,55 ở mạng khác – "phản bội" khi đổi mạng.

In [2]:
import os, sys
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.data import load_all, make_splits, common_features
from tadp.feature_shift import feature_tables, stable_direction
B = "/content/drive/MyDrive/TADP_Q1"
DOMAINS_CIC = {"CIC-IDS2017": ["2017"], "CSE-CIC-IDS2018": ["2018"], "CIC-ToN-IoT": ["ton"], "CIC-UNSW-NB15": ["unsw"]}
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b8_do_lech_dac_trung",
                 DOMAINS=DOMAINS_CIC, SPLIT_MODE="episode")
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200)

data = load_all(cfg)
splits, _ = make_splits(cfg, data)
cols = common_features(data, "no_id")
print(f"\nSố đặc trưng dùng (tập no_id): {len(cols)}")
K, A = feature_tables(cfg, data, splits, cols)
K.to_csv(f"{cfg['RESULT_DIR']}/b8_ks.csv")
A.to_csv(f"{cfg['RESULT_DIR']}/b8_auroc_don_bien.csv")

print("\n15 đặc trưng LỆCH NHIỀU NHẤT (KS trung bình 6 cặp mạng):")
print(K[["ks_all_tb", "ks_benign_tb"]].head(15).round(3).to_string())
print("\n15 đặc trưng LỆCH ÍT NHẤT:")
print(K[["ks_all_tb", "ks_benign_tb"]].tail(15).round(3).to_string())
print(f"\nĐặc trưng ĐẢO CHIỀU giữa các mạng: {int(A['dao_chieu'].sum())} / {len(A)}")
print(A[A["dao_chieu"]].drop(columns=["dao_chieu"]).round(3).to_string())
names = list(data)
for t in names:
    print(f"Đích = {t:16s}: {len(stable_direction(A, [s for s in names if s != t]))} đặc trưng cùng chiều ở 3 mạng nguồn")
print(f"Cùng chiều ở cả 4 mạng (tham khảo): {len(stable_direction(A, names))}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CIC-IDS2017: 999,712 dòng, 87 cột, tỷ lệ tấn công = 0.2411
CSE-CIC-IDS2018: 999,829 dòng, 87 cột, tỷ lệ tấn công = 0.0559
CIC-ToN-IoT: 1,000,028 dòng, 82 cột, tỷ lệ tấn công = 0.5302
CIC-UNSW-NB15: 1,000,399 dòng, 82 cột, tỷ lệ tấn công = 0.0251

Số đặc trưng dùng (tập no_id): 77

15 đặc trưng LỆCH NHIỀU NHẤT (KS trung bình 6 cặp mạng):
                        ks_all_tb  ks_benign_tb
dac_trung                                      
Idle Max                    0.586         0.592
Idle Mean                   0.586         0.591
Idle Min                    0.559         0.554
Subflow Fwd Bytes           0.521         0.595
Subflow Bwd Bytes           0.520         0.576
Bwd Init Win Bytes          0.504         0.541
Fwd Seg Size Min            0.437         0.481
Flow IAT Min                0.417         0.451
Down/Up Ratio               0.378         0.350
Flow

## Ô 3 – Bước 3 và 4: huấn luyện trên 5 tập đặc trưng

Mỗi dòng in ra: cách tiền xử lý, mạng nguồn → mạng đích, tập đặc trưng, số đặc trưng, AUROC trên mạng đích, Macro-F1 ở ngưỡng nguồn. Để tiết kiệm thời gian, mỗi chuyên gia học trên 200.000 luồng lấy mẫu từ phần huấn luyện, với 200 cây.

In [3]:
import os, sys
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.data import load_all, make_splits, common_features
from tadp.feature_shift import run_feature_experiment
B = "/content/drive/MyDrive/TADP_Q1"
DOMAINS_CIC = {"CIC-IDS2017": ["2017"], "CSE-CIC-IDS2018": ["2018"], "CIC-ToN-IoT": ["ton"], "CIC-UNSW-NB15": ["unsw"]}
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b8_do_lech_dac_trung",
                 DOMAINS=DOMAINS_CIC, SPLIT_MODE="episode", LGBM_ROUNDS=200)
data = load_all(cfg)
splits, _ = make_splits(cfg, data)
cols = common_features(data, "no_id")
A = pd.read_csv(f"{cfg['RESULT_DIR']}/b8_auroc_don_bien.csv", index_col=0)   # bảng của Ô 2
run_feature_experiment(cfg, data, splits, cols, A, preprocs=("P1_log", "P3_rank"), k_drop=20, margin=0.05)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CIC-IDS2017: 999,712 dòng, 87 cột, tỷ lệ tấn công = 0.2411
CSE-CIC-IDS2018: 999,829 dòng, 87 cột, tỷ lệ tấn công = 0.0559
CIC-ToN-IoT: 1,000,028 dòng, 82 cột, tỷ lệ tấn công = 0.5302
CIC-UNSW-NB15: 1,000,399 dòng, 82 cột, tỷ lệ tấn công = 0.0251
   [P1_log] CSE-CIC-IDS2018  → CIC-IDS2017      A_tat_ca               77 đặc trưng | AUROC đích = 0.934 | Macro-F1 = 0.639 | 45 giây
   [P1_log] CSE-CIC-IDS2018  → CIC-IDS2017      B_bo_lech_nhieu        57 đặc trưng | AUROC đích = 0.912 | Macro-F1 = 0.670 | 37 giây
   [P1_log] CSE-CIC-IDS2018  → CIC-IDS2017      C_on_dinh_chieu        13 đặc trưng | AUROC đích = 0.964 | Macro-F1 = 0.594 | 27 giây
   [P1_log] CSE-CIC-IDS2018  → CIC-IDS2017      D_B_giao_C             11 đặc trưng | AUROC đích = 0.964 | Macro-F1 = 0.591 | 23 giây
   [P1_log] CSE-CIC-IDS2018  → CIC-IDS2017      E_tham_khao_nhan_dich   5 đặc trưng | AUR

,preproc,source,target,tap,so_dac_trung,auroc_trong_mien,auroc,ap,macro_f1_thr_src,macro_f1_oracle_thr,macro_f1_constant
0,P1_log,CSE-CIC-IDS2018,CIC-IDS2017,A_tat_ca,77,0.999975,0.934467,0.823383,0.638556,0.852720,0.431467
1,P1_log,CSE-CIC-IDS2018,CIC-IDS2017,B_bo_lech_nhieu,57,0.999943,0.911564,0.813487,0.669807,0.839616,0.431467
2,P1_log,CSE-CIC-IDS2018,CIC-IDS2017,C_on_dinh_chieu,13,0.999914,0.964227,0.903285,0.593690,0.895016,0.431467
3,P1_log,CSE-CIC-IDS2018,CIC-IDS2017,D_B_giao_C,11,0.999964,0.964271,0.907778,0.591434,0.906266,0.431467
4,P1_log,CSE-CIC-IDS2018,CIC-IDS2017,E_tham_khao_nhan_dich,5,0.999535,0.693121,0.665518,0.687742,0.825065,0.431467
5,P1_log,CIC-ToN-IoT,CIC-IDS2017,A_tat_ca,77,0.999976,0.424352,0.204160,0.383283,0.492557,0.431467
6,P1_log,CIC-ToN-IoT,CIC-IDS2017,B_bo_lech_nhieu,57,0.999626,0.344181,0.181389,0.352473,0.438015,0.431467
7,P1_log,CIC-ToN-IoT,CIC-IDS2017,C_on_dinh_chieu,13,0.999619,0.515736,0.250296,0.468193,0.541469,0.431467
8,P1_log,CIC-ToN-IoT,CIC-IDS2017,D_B_giao_C,10,0.997266,0.510903,0.385639,0.486558,0.615144,0.431467
9,P1_log,CIC-ToN-IoT,CIC-IDS2017,E_tham_khao_nhan_dich,5,0.973392,0.522815,0.371191,0.319617,0.715210,0.431467


## Ô 4 – Tổng hợp

Bảng 8.1: trung bình 12 cặp cho mỗi (cách tiền xử lý, tập đặc trưng). Bảng 8.2: AUROC từng cặp. Chạy xong báo Claude.

In [4]:
import sys
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.feature_shift import summarize_feature_experiment
B = "/content/drive/MyDrive/TADP_Q1"
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b8_do_lech_dac_trung")
summarize_feature_experiment(cfg)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BẢNG 8.1 – Trung bình 12 cặp nguồn → đích, theo cách tiền xử lý và tập đặc trưng
                                so_dac_trung  auroc_trong_mien  auroc  macro_f1_thr_src  macro_f1_oracle_thr  macro_f1_constant  ty_le_dao_chieu
preproc tap                                                                                                                                    
P1_log  A_tat_ca                     77.000             0.999  0.540             0.467                0.572              0.439            0.417
        B_bo_lech_nhieu              57.000             0.997  0.575             0.478                0.561              0.439            0.333
        C_on_dinh_chieu               9.250             0.998  0.496             0.419                0.553              0.439            0.500
        D_B_giao_C                    6.583             0.994  0.517 

(                               so_dac_trung  auroc_trong_mien  auroc  macro_f1_thr_src  macro_f1_oracle_thr  macro_f1_constant  ty_le_dao_chieu
 preproc tap                                                                                                                                    
 P1_log  A_tat_ca                     77.000             0.999  0.540             0.467                0.572              0.439            0.417
         B_bo_lech_nhieu              57.000             0.997  0.575             0.478                0.561              0.439            0.333
         C_on_dinh_chieu               9.250             0.998  0.496             0.419                0.553              0.439            0.500
         D_B_giao_C                    6.583             0.994  0.517             0.485                0.583              0.439            0.583
         E_tham_khao_nhan_dich         5.000             0.988  0.463             0.438                0.543              0.439   